# ADAN Trading Bot — Colab Training Pipeline

This notebook runs the full ADAN training pipeline on Google Colab:

1. **Clone** the repository (with submodules)
2. **Install** dependencies
3. **Generate** the training dataset (6 months of BTCUSDT + XRPUSDT via ccxt)
4. **Train** with Ray Tune Population-Based Training (PBT)
5. **Download** the trained model

> **GPU**: Select *Runtime → Change runtime type → T4 GPU*

## 1. Clone Repository

In [ ]:
import os

# Clone if not already present
if not os.path.exists('ADAN'):
    !git clone https://github.com/Cabrel10/ADAN.git
    %cd ADAN
    !git submodule update --init --recursive
else:
    %cd ADAN
    !git pull origin main
    !git submodule update --recursive

print('\n=== Repository ready ===')
!ls -la

## 2. Install Dependencies

In [ ]:
%cd /content/ADAN/bot

# Install core requirements
!pip install -q -r requirements.txt 2>/dev/null || echo 'requirements.txt not found, installing manually'

# Install additional dependencies for training
!pip install -q \
    'ray[tune]>=2.9' \
    pandas_ta \
    ccxt \
    python-dotenv \
    stable-baselines3 \
    gymnasium \
    torch \
    tqdm \
    plotly \
    rich

# Verify key imports
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import ray
print(f'Ray: {ray.__version__}')

print('\n=== Dependencies installed ===')

## 3. Generate Training Dataset

Downloads 6 months of OHLCV data from Binance via ccxt and computes
the exact indicators defined in `config/config.yaml`.

**This takes ~5-10 minutes** (rate-limited API calls).

To use fewer candles for a quick test, set `ADAN_CANDLES=500` before running.

In [ ]:
%cd /content/ADAN/bot

# Optional: reduce candle count for quick test
# import os; os.environ['ADAN_CANDLES'] = '500'

!python scripts/generate_colab_dataset.py

# Verify the generated files
print('\n=== Generated Parquet Files ===')
!find data/processed/indicators -name '*.parquet' -exec ls -lh {} \;

## 4. Train with Ray Tune PBT

Launches Population-Based Training with 4 PPO workers.

- **Steps**: 2,000,000 total timesteps (adjust as needed)
- **Workers**: 4 concurrent trials
- **Sub-envs**: 1 per worker (DummyVecEnv for Colab memory)

> Training on T4 GPU takes ~2-4 hours for 2M steps.

In [ ]:
%cd /content/ADAN/bot

# Create .env file for dotenv loading
!cp .env.example .env 2>/dev/null || echo 'TRADING_MODE=paper' > .env

# Run PBT training
# --no-subproc: use DummyVecEnv (safer on Colab)
# --num-cpus 2: Colab has 2 CPUs
# --num-samples 4: 4 PBT trials
# --envs-per-worker 1: 1 env per trial (memory-safe)
!PYTHONPATH=src:$PYTHONPATH python scripts/train_parallel_agents.py \
    --config config/config.yaml \
    --steps 2000000 \
    --num-cpus 2 \
    --num-samples 4 \
    --envs-per-worker 1 \
    --no-subproc \
    --steps-per-iter 5000

## 5. Download Trained Model

In [ ]:
import zipfile
import glob
import os

%cd /content/ADAN/bot

# Find all model checkpoints
model_files = glob.glob('logs/ray_results/**/*.zip', recursive=True)
vec_files = glob.glob('logs/ray_results/**/*.pkl', recursive=True)
state_files = glob.glob('logs/ray_results/**/worker_state.json', recursive=True)
summary_files = glob.glob('logs/ray_results/**/pbt_summary.json', recursive=True)

all_files = model_files + vec_files + state_files + summary_files
print(f'Found {len(all_files)} files to package:')
for f in all_files[:10]:
    print(f'  {f}')

# Create zip archive
zip_path = '/content/ADAN_trained_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in all_files:
        arcname = os.path.relpath(fpath, '/content/ADAN/bot')
        zf.write(fpath, arcname)
        print(f'  Added: {arcname}')

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'\n=== Archive: {zip_path} ({size_mb:.1f} MB) ===')

# Download to local machine
try:
    from google.colab import files
    files.download(zip_path)
    print('Download started!')
except ImportError:
    print('Not running on Colab - file saved at:', zip_path)